In [0]:
import os
import shutil
import zipfile

ZIP_PATH = "/Volumes/workspace/bronze/landing/landing_upload.zip"
TARGET = "/Volumes/workspace/bronze/landing/files"

written = skipped = total_bytes = 0
with zipfile.ZipFile(ZIP_PATH) as zf:
    for info in zf.infolist():
        # Windows zips can store paths with backslashes; normalise to forward slashes
        name = info.filename.replace("\\", "/")
        if name.endswith("/"):
            continue  # folder entry, nothing to write
        dest = os.path.join(TARGET, name)
        total_bytes += info.file_size
        if os.path.exists(dest) and os.path.getsize(dest) == info.file_size:
            skipped += 1  # already unpacked by an earlier run
            continue
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        with zf.open(info) as src, open(dest, "wb") as out:
            shutil.copyfileobj(src, out)
        written += 1

print(f"written {written}, skipped {skipped}, total {written + skipped} files, "
      f"{total_bytes / 1024**2:.0f} MB")
print(sorted(os.listdir(TARGET)))

written 16346, skipped 0, total 16346 files, 293 MB
['carbon_intensity', 'elexon_demand_forecast', 'elexon_demand_outturn', 'elexon_generation_by_fuel', 'elexon_system_prices', 'elexon_wind_forecast', 'elexon_wind_solar_actual', 'elexon_wind_solar_day_ahead', 'open_meteo_ecmwf']


In [0]:
import os
import shutil
import zipfile

LANDING = "/Volumes/workspace/bronze/landing"
OM = f"{LANDING}/files/open_meteo_ecmwf"
QUARANTINE = f"{LANDING}/quarantine/open_meteo_ecmwf"

# 1. Set the two hollow days aside, exactly as on the laptop (moved, never deleted)
for day in ["2025-08-08", "2026-06-24"]:
    src, dst = f"{OM}/{day}", f"{QUARANTINE}/{day}"
    if os.path.exists(src):
        dbutils.fs.mv(src, dst, recurse=True)
        print("quarantined", day)

# 2. Unpack the 4 rescued days
with zipfile.ZipFile(f"{LANDING}/weather_fix.zip") as zf:
    for info in zf.infolist():
        name = info.filename.replace("\\", "/")
        if name.endswith("/"):
            continue
        dest = os.path.join(OM, name)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        with zf.open(info) as src, open(dest, "wb") as out:
            shutil.copyfileobj(src, out)
        print("wrote", name)

# 3. Check: what's in each affected day folder now
for day in ["2025-08-06", "2025-08-07", "2025-08-08", "2025-08-09", "2025-08-10", "2026-06-24"]:
    path = f"{OM}/{day}"
    print(day, sorted(os.listdir(path)) if os.path.exists(path) else "no folder (gap)")

quarantined 2025-08-08
quarantined 2026-06-24
wrote 2025-08-06/20261018T224356041971Z_3d9fae2f.json
wrote 2025-08-06/20261018T224356041971Z_3d9fae2f.meta.json
wrote 2025-08-07/20261018T224359578332Z_665bf1fa.json
wrote 2025-08-07/20261018T224359578332Z_665bf1fa.meta.json
wrote 2025-08-10/20261018T224412582155Z_e8da63d8.json
wrote 2025-08-10/20261018T224412582155Z_e8da63d8.meta.json
wrote 2026-06-24/20261018T224514776128Z_0c68f151.json
wrote 2026-06-24/20261018T224514776128Z_0c68f151.meta.json
2025-08-06 ['20261018T224356041971Z_3d9fae2f.json', '20261018T224356041971Z_3d9fae2f.meta.json']
2025-08-07 ['20261018T224359578332Z_665bf1fa.json', '20261018T224359578332Z_665bf1fa.meta.json']
2025-08-08 []
2025-08-09 no folder (gap)
2025-08-10 ['20261018T224412582155Z_e8da63d8.json', '20261018T224412582155Z_e8da63d8.meta.json']
2026-06-24 ['20261018T224514776128Z_0c68f151.json', '20261018T224514776128Z_0c68f151.meta.json']


In [0]:
import json
import os

OM = "/Volumes/workspace/bronze/landing/files/open_meteo_ecmwf"

# Remove the empty folder left behind by the move (only if it really is empty)
empty = f"{OM}/2025-08-08"
if os.path.isdir(empty) and not os.listdir(empty):
    os.rmdir(empty)
    print("removed empty folder 2025-08-08")

# Show one metadata file: a rescued day, and a normal day for comparison
for day in ["2025-08-07", "2025-10-26"]:
    meta = [f for f in os.listdir(f"{OM}/{day}") if f.endswith(".meta.json")][0]
    with open(f"{OM}/{day}/{meta}") as f:
        print(day, json.dumps(json.load(f), indent=2))

removed empty folder 2025-08-08
2025-08-07 {
  "source": "open_meteo_ecmwf",
  "partition": "2025-08-07",
  "url": "https://single-runs-api.open-meteo.com/v1/forecast?latitude=53.9%2C54.8%2C52.2%2C56.6%2C58.2%2C55.7%2C52.3%2C51.2%2C51.1%2C51.5%2C52.48%2C53.48&longitude=1.8%2C1.9%2C2.5%2C-1.8%2C-2.7%2C-4.3%2C0.1%2C0.8%2C-2.9%2C-0.13%2C-1.9%2C-2.24&run=2025-08-04T06%3A00&models=ecmwf_ifs&hourly=temperature_2m%2Cshortwave_radiation%2Cwind_speed_100m%2Ccloud_cover&forecast_days=5&timezone=GMT",
  "params": {
    "latitude": "53.9,54.8,52.2,56.6,58.2,55.7,52.3,51.2,51.1,51.5,52.48,53.48",
    "longitude": "1.8,1.9,2.5,-1.8,-2.7,-4.3,0.1,0.8,-2.9,-0.13,-1.9,-2.24",
    "run": "2025-08-04T06:00",
    "models": "ecmwf_ifs",
    "hourly": "temperature_2m,shortwave_radiation,wind_speed_100m,cloud_cover",
    "forecast_days": "5",
    "timezone": "GMT"
  },
  "status_code": 200,
  "fetched_at": "2026-10-18T22:43:59.578332+00:00",
  "sha256": "665bf1fa2a199d9b4cb0134c43b905201e1c3a06478e0fac7254ad